# LV4 - Neuronske mreže (2/4)
## Zadatak 1 - Binarna klasifikacija - Spam filter

### a) Učitavanje skupa podataka

In [ ]:
import pandas as pd

df_spam = pd.read_csv('SpamDetectionData.txt')
print("Prvih 3 podatka:")
print(df_spam.head(3))
print("\nKolone:", df_spam.columns.tolist())

### b) Izdvajanje X i y, uklanjanje HTML tagova

In [ ]:
X_spam = df_spam['Message'].values
y_spam = df_spam['Label'].values

# Uklanjanje <p> i </p> tagova
X_spam = [msg.replace('<p>', '').replace('</p>', '') for msg in X_spam]

print("Prva rečenica:", X_spam[0][:100], '...')
print("\nBroj slova prve rečenice:", len(X_spam[0]))
print("Broj slova druge rečenice:", len(X_spam[1]))

### c) Podjela na trening i testni skup (10% test)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_spam, X_test_spam, y_train_spam, y_test_spam = train_test_split(
    X_spam, y_spam, test_size=0.1, random_state=42
)

print("Veličina trening skupa:", len(X_train_spam))
print("Veličina testnog skupa:", len(X_test_spam))

### d) Tokenizacija teksta

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer_spam = Tokenizer()
tokenizer_spam.fit_on_texts(X_train_spam)

train_data_seq_spam = tokenizer_spam.texts_to_sequences(X_train_spam)
test_data_seq_spam  = tokenizer_spam.texts_to_sequences(X_test_spam)

# Tri najčešće riječi
word_counts = tokenizer_spam.word_counts
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
print("Tri najčešće riječi:", sorted_words[:3])

print("\nPrva rečenica iz trening skupa:")
print(X_train_spam[0][:200])
print("\nNiz cijelih brojeva za prvu rečenicu:")
print(train_data_seq_spam[0][:30], '...')

### e) Vektorizacija sekvenci i labela

In [ ]:
import numpy as np

def vectorize_sequences(sequences, dimension=4000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

def vectorize_labels(labels):
    results = np.zeros(len(labels))
    for i, label in enumerate(labels):
        if label.lower() == 'spam':
            results[i] = 1
    return results

x_train_spam_vec = vectorize_sequences(train_data_seq_spam, dimension=4000)
x_test_spam_vec  = vectorize_sequences(test_data_seq_spam,  dimension=4000)

y_train_spam_vec = vectorize_labels(y_train_spam)
y_test_spam_vec  = vectorize_labels(y_test_spam)

print("Oblik trening skupa nakon vektorizacije:", x_train_spam_vec.shape)
print("Oblik testnog skupa nakon vektorizacije:", x_test_spam_vec.shape)

### f) Definisanje Keras modela

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model_spam = Sequential([
    Dense(8, activation='relu', input_shape=(4000,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_spam.summary()

### g) Kompajliranje modela

In [ ]:
model_spam.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### h) Treniranje modela i grafički prikaz

In [ ]:
import matplotlib.pyplot as plt

history_spam = model_spam.fit(
    x_train_spam_vec, y_train_spam_vec,
    epochs=5,
    batch_size=128,
    validation_split=0.3
)

print("Tačnost (posljednja epoha):", history_spam.history['accuracy'][-1])
print("Gubitak (posljednja epoha):", history_spam.history['loss'][-1])

# Grafički prikaz gubitka
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_spam.history['loss'],     label='Training loss')
ax1.plot(history_spam.history['val_loss'], label='Validation loss')
ax1.set_title('Training and validation loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history_spam.history['accuracy'],     label='Training acc')
ax2.plot(history_spam.history['val_accuracy'], label='Validation acc')
ax2.set_title('Training and validation accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

### i) Evaluacija modela nad testnim skupom

In [ ]:
loss_spam, acc_spam = model_spam.evaluate(x_test_spam_vec, y_test_spam_vec)
print(f"Tačnost na testnom skupu: {acc_spam:.4f}")
print(f"Gubitak na testnom skupu: {loss_spam:.4f}")

### j) Klasifikacija proizvoljne poruke

In [ ]:
custom_message = ["Congratulations! You have won a free prize. Click here to claim your reward now!"]

custom_seq = tokenizer_spam.texts_to_sequences(custom_message)
custom_vec = vectorize_sequences(custom_seq, dimension=4000)

prediction = model_spam.predict(custom_vec)[0][0]
label = 'Spam' if prediction >= 0.5 else 'Ham'

print(f"Poruka: {custom_message[0]}")
print(f"Vjerovatnoća spam-a: {prediction:.4f}")
print(f"Klasifikacija: {label}")

---
## Zadatak 2 - Višeklasna klasifikacija - klasificiranje Stackoverflow pitanja

### a) Učitavanje stackoverflow.csv i prikaz posljednjih 3 podatka

In [ ]:
df_so = pd.read_csv('stackoverflow.csv')
print("Posljednja 3 podatka:")
print(df_so.tail(3))

### b) Izdvajanje X i y, broj jedinstvenih labela

In [ ]:
X_so = df_so['post'].values
y_so = df_so['tags'].values

unique_labels = df_so['tags'].unique()
print("Broj jedinstvenih labela (programskih jezika):", len(unique_labels))
print("Programski jezici:", unique_labels)

### c) One-hot enkodiranje labela

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
y_so_encoded = label_encoder.fit_transform(y_so)

y_so_categorical = to_categorical(y_so_encoded)

print("One-hot enkodiran niz labela (prvih 5):")
print(y_so_categorical[:5])
print("\nOblik labela:", y_so_categorical.shape)

### d) Podjela na trening i testni skup (10% test)

In [ ]:
X_train_so, X_test_so, y_train_so, y_test_so = train_test_split(
    X_so, y_so_categorical, test_size=0.1, random_state=42
)

print("Veličina trening skupa:", len(X_train_so))
print("Veličina testnog skupa:", len(X_test_so))

### e) Tokenizacija i vektorizacija (500 najčešćih riječi)

In [ ]:
VOCAB_SIZE = 500

tokenizer_so = Tokenizer(num_words=VOCAB_SIZE)
tokenizer_so.fit_on_texts(X_train_so)

train_data_seq_so = tokenizer_so.texts_to_sequences(X_train_so)
test_data_seq_so  = tokenizer_so.texts_to_sequences(X_test_so)

x_train_so_vec = vectorize_sequences(train_data_seq_so, dimension=VOCAB_SIZE)
x_test_so_vec  = vectorize_sequences(test_data_seq_so,  dimension=VOCAB_SIZE)

print("Oblik trening skupa:", x_train_so_vec.shape)
print("Oblik testnog skupa:", x_test_so_vec.shape)

### f) Definisanje Keras modela za višeklasnu klasifikaciju

In [ ]:
num_classes = y_so_categorical.shape[1]

model_so = Sequential([
    Dense(32, activation='relu', input_shape=(VOCAB_SIZE,)),
    Dense(8,  activation='relu'),
    Dense(num_classes, activation='softmax')
])

### g) Kompajliranje modela i prikaz sažetka

In [ ]:
model_so.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_so.summary()

### h) Treniranje modela i grafički prikaz

In [ ]:
history_so = model_so.fit(
    x_train_so_vec, y_train_so,
    epochs=8,
    batch_size=8,
    validation_split=0.25
)

print("Tačnost (posljednja epoha):", history_so.history['accuracy'][-1])
print("Gubitak (posljednja epoha):", history_so.history['loss'][-1])

# Grafički prikaz
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_so.history['loss'],     label='Training loss')
ax1.plot(history_so.history['val_loss'], label='Validation loss')
ax1.set_title('Training and validation loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history_so.history['accuracy'],     label='Training acc')
ax2.plot(history_so.history['val_accuracy'], label='Validation acc')
ax2.set_title('Training and validation accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

### i) Evaluacija modela nad testnim skupom

In [ ]:
loss_so, acc_so = model_so.evaluate(x_test_so_vec, y_test_so)
print(f"Tačnost na testnom skupu: {acc_so:.4f}")
print(f"Gubitak na testnom skupu: {loss_so:.4f}")